# cc-1.1 — Act-PRM × tau2 retail: results

Training/eval curves for the three-stage pipeline, reconstructed from the
`metrics.jsonl` that `tinker_cookbook.utils.ml_log` writes under each run's
`--log_path` (one row per step; the eval branch flushes an extra row per eval —
we dedupe on `progress/batch`, keeping the last).

- **Stage 1 — Act-PRM EM** (thought generation): reward = length-penalized
  `p(x | s, z)` -> `*/try_0/final_reward`.
- **Stage 2 — SFT** (4 datasets): `train/loss`, and the offline eval metrics
  `eval/eval_action_ppl` (action-token perplexity) + `eval/eval_action_accuracy`.
- **Stage 3 — env RL** (from the best Stage-2 ckpt): reward `*/try_0/final_reward`.

Runnable in the `./.venv` uv kernel (`uv run jupyter lab`). Point the config cell
at real run dirs; with the placeholder paths below every cell still runs (loaders
return empty frames and plots are skipped with a note).

In [ ]:
# === CONFIG - EDIT THESE PATHS (placeholders; safe to run as-is) =============
# A "run dir" is either a metrics.jsonl file or any parent dir containing one
# (we glob **/metrics.jsonl and take the most recently modified).
from pathlib import Path

LOGS = Path("logs")  # --log_path root

# Stage 1 - Act-PRM EM thought generation (on-policy + base-scored)
STAGE1_RUNS = {
    "policy-score": LOGS / "act_prm_tau2_retail" / "PLACEHOLDER_stage1_policy",
    "base-score":   LOGS / "act_prm_tau2_retail" / "PLACEHOLDER_stage1_base",
}

# Stage 2 - SFT, one dir per variant (the 4 datasets)
SFT_RUNS = {
    "actions_only":    LOGS / "act_prm_tau2_retail" / "PLACEHOLDER_sft_actions_only",
    "thoughts_policy": LOGS / "act_prm_tau2_retail" / "PLACEHOLDER_sft_thoughts_policy",
    "thoughts_base":   LOGS / "act_prm_tau2_retail" / "PLACEHOLDER_sft_thoughts_base",
    "expert_thoughts": LOGS / "act_prm_tau2_retail" / "PLACEHOLDER_sft_expert_thoughts",
}

# Stage 3 - env RL from each Stage-2 best ckpt
STAGE3_RUNS = {
    "thoughts_policy": LOGS / "act_prm_tau2_retail" / "PLACEHOLDER_rl_thoughts_policy",
    "thoughts_base":   LOGS / "act_prm_tau2_retail" / "PLACEHOLDER_rl_thoughts_base",
}


In [ ]:
# === IMPORTS + HELPERS ======================================================
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "figure.dpi": 110})

# Brand-neutral, colorblind-safe categorical palette (swap for your own).
PALETTE = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756", "#72B7B2"]


def find_metrics(path):
    """Resolve a run dir/file to a metrics.jsonl path (newest if several)."""
    p = Path(path)
    if p.is_file():
        return p
    if p.is_dir():
        hits = sorted(p.glob("**/metrics.jsonl"), key=lambda x: x.stat().st_mtime)
        return hits[-1] if hits else None
    return None


def load_metrics(path) -> pd.DataFrame:
    """Load a run's metrics.jsonl into a DataFrame, deduped on progress/batch
    (keep last - the eval flush writes a partial row before the full one).
    Returns an empty frame (with a warning) if nothing is found."""
    mp = find_metrics(path)
    if mp is None:
        warnings.warn(f"no metrics.jsonl under {path!r} - skipping")
        return pd.DataFrame()
    rows = [json.loads(l) for l in open(mp) if l.strip()]
    df = pd.DataFrame(rows)
    if "progress/batch" in df:
        df = df.drop_duplicates("progress/batch", keep="last")
        df = df.sort_values("progress/batch").reset_index(drop=True)
    return df


def load_group(runs: dict) -> dict:
    return {label: load_metrics(p) for label, p in runs.items()}


def _xy(df, col, xcol="progress/batch"):
    """(x, y) for a metric column, dropping rows where it's NaN/absent."""
    if df.empty or col not in df:
        return None, None
    cols = [c for c in (xcol, col) if c in df]
    sub = df[cols].dropna()
    x = sub[xcol] if xcol in sub else np.arange(len(sub))
    return x, sub[col]


def plot_metric(dfs: dict, col: str, title: str, ylabel: str, ax=None):
    """Overlay one metric across labeled runs. Skips runs missing the column."""
    ax = ax or plt.gca()
    plotted = 0
    for i, (label, df) in enumerate(dfs.items()):
        x, y = _xy(df, col)
        if x is None or len(y) == 0:
            continue
        ax.plot(x, y, marker="o", ms=3, lw=1.6, color=PALETTE[i % len(PALETTE)], label=label)
        plotted += 1
    ax.set(title=title, xlabel="step (progress/batch)", ylabel=ylabel)
    if plotted:
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "no data\n(set real run dirs in CONFIG)",
                ha="center", va="center", transform=ax.transAxes, color="gray")
    return ax

print("helpers defined: find_metrics, load_metrics, load_group, plot_metric")


## Stage 1 - Act-PRM EM: reward `p(x | s, z)`

In [ ]:
s1 = load_group(STAGE1_RUNS)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_metric(s1, "train/try_0/final_reward", "Stage 1 train reward", "p(x|s,z)", axes[0])
plot_metric(s1, "eval/try_0/final_reward",  "Stage 1 eval reward",  "p(x|s,z)", axes[1])
plt.tight_layout(); plt.show()


## Stage 2 - SFT: train loss, eval action-PPL, eval action accuracy

In [ ]:
sft = load_group(SFT_RUNS)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_metric(sft, "train/loss",                "SFT train loss",      "CE loss",        axes[0])
plot_metric(sft, "eval/eval_action_ppl",      "SFT eval action-PPL", "perplexity",     axes[1])
plot_metric(sft, "eval/eval_action_accuracy", "SFT eval action-acc", "token accuracy", axes[2])
plt.tight_layout(); plt.show()


## Stage 2 - 4-way SFT comparison

Best (early-stop) eval action-PPL and accuracy per dataset. Uses
`eval/eval_action_ppl_best` when present (written when `best_metric:
eval_action_ppl`), else the min/max over the eval curve.

In [ ]:
def best_ppl(df):
    if df.empty: return np.nan
    if "eval/eval_action_ppl_best" in df:
        v = df["eval/eval_action_ppl_best"].dropna()
        if len(v): return float(v.iloc[-1])
    return float(df["eval/eval_action_ppl"].min()) if "eval/eval_action_ppl" in df else np.nan

def best_acc(df):
    if df.empty or "eval/eval_action_accuracy" not in df: return np.nan
    return float(df["eval/eval_action_accuracy"].max())

labels = list(SFT_RUNS)
ppls = [best_ppl(sft[l]) for l in labels]
accs = [best_acc(sft[l]) for l in labels]
summary = pd.DataFrame({"best_eval_action_ppl": ppls, "best_eval_action_accuracy": accs}, index=labels)
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = [PALETTE[i % len(PALETTE)] for i in range(len(labels))]
axes[0].bar(labels, ppls, color=colors); axes[0].set(title="Best eval action-PPL (lower=better)", ylabel="perplexity")
axes[1].bar(labels, accs, color=colors); axes[1].set(title="Best eval action accuracy (higher=better)", ylabel="token accuracy")
for ax in axes: ax.tick_params(axis="x", rotation=20)
if all(np.isnan(ppls)):
    axes[0].text(0.5, 0.5, "no data (set real SFT run dirs)", ha="center", va="center", transform=axes[0].transAxes, color="gray")
plt.tight_layout(); plt.show()


## Stage 3 - env RL from the best Stage-2 ckpt: reward

In [ ]:
s3 = load_group(STAGE3_RUNS)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_metric(s3, "train/try_0/final_reward", "Stage 3 train reward", "reward", axes[0])
plot_metric(s3, "eval/try_0/final_reward",  "Stage 3 eval reward",  "reward", axes[1])
plt.tight_layout(); plt.show()
